In [3]:
from getpass import getpass
import pandas as pd
import networkx as nx
from neo4j import GraphDatabase

mdp = getpass("Mot de passe Neo4j : ")
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", mdp))
def run(q):
    with driver.session() as s:
        return pd.DataFrame(s.run(q).data())

users = run("MATCH (u:User) RETURN u.userId AS user, u.louvain AS louvain, u.lpa AS lpa")
edges = run("""MATCH (a:User)-[:IS_FRIEND]-(b:User) WHERE a.userId < b.userId
               RETURN a.userId AS a, b.userId AS b""")
G = nx.Graph()
G.add_nodes_from(users["user"])
G.add_edges_from(zip(edges["a"], edges["b"]))

for method in ["louvain", "lpa"]:
    part = [set(g["user"]) for _, g in users.groupby(method)]
    sizes = sorted((len(c) for c in part), reverse=True)
    print(method, "| communautés :", len(part),
          "| modularité :", round(nx.community.modularity(G, part), 4),
          "| 5 plus grandes :", sizes[:5])

louvain | communautés : 30 | modularité : 0.4542 | 5 plus grandes : [565, 374, 344, 257, 235]
lpa | communautés : 32 | modularité : 0.0225 | 5 plus grandes : [1785, 11, 7, 7, 6]


In [4]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(users["louvain"], users["lpa"])
nmi = normalized_mutual_info_score(users["louvain"], users["lpa"])
print(f"ARI = {ari:.3f}   NMI = {nmi:.3f}")

# Pour chaque grande communauté Louvain : où vont ses membres dans LPA ?
for c in users["louvain"].value_counts().head(5).index:
    members = users[users["louvain"] == c]
    lpa_counts = members["lpa"].value_counts()
    print(f"Louvain {c} ({len(members)} users) -> LPA {lpa_counts.index[0]} : "
          f"{100 * lpa_counts.iloc[0] / len(members):.1f} % des membres "
          f"| réparti dans {lpa_counts.size} communauté(s) LPA")

ARI = 0.037   NMI = 0.266
Louvain 1357 (565 users) -> LPA 17642 : 96.5 % des membres | réparti dans 5 communauté(s) LPA
Louvain 373 (374 users) -> LPA 17642 : 100.0 % des membres | réparti dans 1 communauté(s) LPA
Louvain 689 (344 users) -> LPA 17642 : 100.0 % des membres | réparti dans 1 communauté(s) LPA
Louvain 74 (257 users) -> LPA 17642 : 96.1 % des membres | réparti dans 4 communauté(s) LPA
Louvain 1061 (235 users) -> LPA 17642 : 100.0 % des membres | réparti dans 1 communauté(s) LPA


In [5]:
listens = run("""MATCH (u:User)-[:LISTEN_TO]->(a:Artist)
                 RETURN u.userId AS user, a.name AS artist""")
tags = run("""MATCH (u:User)-[r:TAGGED]->(:Artist)
              RETURN u.userId AS user, r.tag AS tag""")

N = len(users)
comm = users.set_index("user")["louvain"]
listens["louvain"] = listens["user"].map(comm)
tags["louvain"] = tags["user"].map(comm)
artist_share_all = listens.groupby("artist")["user"].nunique() / N
tag_share_all = tags.groupby("tag")["user"].nunique() / N

def typical(df, item, share_all, c, n_c, k=8):
    in_c = df[df["louvain"] == c].groupby(item)["user"].nunique()
    share_c = in_c / n_c
    out = pd.DataFrame({"% communauté": (100 * share_c).round(1),
                        "% tous": (100 * share_all[share_c.index]).round(1),
                        "lift": (share_c / share_all[share_c.index]).round(1)})
    return out[share_c >= 0.10].sort_values("lift", ascending=False).head(k)

for c in users["louvain"].value_counts().head(5).index:
    n_c = int((users["louvain"] == c).sum())
    print("=" * 60)
    print(f"COMMUNAUTÉ {c} : {n_c} utilisateurs")
    print("\nArtistes typiques :")
    print(typical(listens, "artist", artist_share_all, c, n_c).to_string())
    print("\nTags typiques :")
    print(typical(tags, "tag", tag_share_all, c, n_c).to_string())

COMMUNAUTÉ 1357 : 565 utilisateurs

Artistes typiques :
                         % communauté  % tous  lift
artist                                             
Escape The Fate                  10.1     3.6   2.8
Megadeth                         15.2     5.5   2.8
Bullet for My Valentine          11.2     4.3   2.6
Judas Priest                     11.7     4.5   2.6
Slayer                           11.0     4.3   2.6
A Day to Remember                10.4     4.1   2.6
KISS                             10.4     4.3   2.4
Rise Against                     10.4     4.3   2.4

Tags typiques :
                   % communauté  % tous  lift
tag                                          
post-hardcore              11.0     4.7   2.4
metalcore                  14.9     6.1   2.4
heavy metal                20.5     8.7   2.4
thrash metal               13.8     5.9   2.4
death metal                11.0     5.1   2.2
screamo                    11.0     5.2   2.1
progressive metal          11.0     5.2

In [6]:
names = {1357: "Metal", 373: "Pop teen & R&B", 689: "Électro expérimentale",
         74: "Synth-pop 80s", 1061: "Pop latine"}
comm = users.set_index("user")["louvain"].to_dict()

bet = nx.betweenness_centrality(G, normalized=True)
rows = []
for u in G.nodes():
    friends = list(G.neighbors(u))
    if not friends:
        continue
    share = pd.Series([comm[f] for f in friends]).value_counts(normalize=True)
    rows.append({
        "user": u,
        "communauté": names.get(comm[u], comm[u]),
        "amis": len(friends),
        "betweenness": round(bet[u], 4),
        "% amis hors communauté": round(100 * sum(comm[f] != comm[u] for f in friends) / len(friends), 1),
        "participation": round(1 - (share ** 2).sum(), 3),
        "répartition des amis": ", ".join(f"{names.get(c, c)} {100*p:.0f}%" for c, p in share.head(3).items()),
    })
bridge = pd.DataFrame(rows).set_index("user")

print("=== Top 10 betweenness (les ponts de la Q7) ===")
print(bridge.sort_values("betweenness", ascending=False).head(10).to_string())
print("\n=== Top 10 participation (au moins 10 amis) ===")
print(bridge[bridge["amis"] >= 10].sort_values("participation", ascending=False).head(10).to_string())
print("\nParticipation moyenne : top 10 betweenness =",
      round(bridge.sort_values("betweenness", ascending=False).head(10)["participation"].mean(), 3),
      "| tout le réseau =", round(bridge["participation"].mean(), 3))

=== Top 10 betweenness (les ponts de la Q7) ===
                 communauté  amis  betweenness  % amis hors communauté  participation                                              répartition des amis
user                                                                                                                                                   
1543                  Metal   119       0.0665                    43.7          0.624            Metal 56%, Électro expérimentale 21%, Synth-pop 80s 9%
1281                  Metal   110       0.0545                    36.4          0.553           Metal 64%, Électro expérimentale 15%, Synth-pop 80s 10%
831           Synth-pop 80s   106       0.0487                    74.5          0.803  Synth-pop 80s 25%, Pop teen & R&B 24%, Électro expérimentale 18%
1258          Synth-pop 80s    86       0.0430                    50.0          0.634           Synth-pop 80s 50%, Metal 29%, Électro expérimentale 17%
78    Électro expérimentale    81       